# TERRA Ablation

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 16.11.2025
- Date of Last Modification: 06.06.2026 (by Sebastian Birk; sb75@sanger.ac.uk)

Run this notebook in the ```rapids-singlecell-terra``` environment:
```
source /etc/profile.d/modules.sh 
module load cellgen/conda 
conda activate /nfs/team361/sb75/ENVS/rapids-singlecell-terra 
```

Before running this notebook, model inference needs to be run using ```../analysis/data_analysis/hst_corpus_110m_analysis.ipynb``` or ```../analysis/benchmarking/terra_benchmarking.ipynb```. 

# 1. Setup

## 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import random

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import rapids_singlecell as rsc
import scanpy as sc
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

## 1.2 Configure Paths

In [ ]:
artifacts_folder_path = '/nfs/team361/sb75/terra-reproducibility/artifacts'
ablation_artifacts_folder_path = f'{artifacts_folder_path}/ablation'

In [ ]:
os.makedirs(f'{ablation_artifacts_folder_path}/figures', exist_ok=True)

## 1.3 Run Notebook Setup

In [ ]:
plt.rcParams.update({
    "font.size": 12,         
    "axes.labelsize": 12,    
    "axes.titlesize": 12,     
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
})
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
sc.settings.figdir = f'{ablation_artifacts_folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=12
)

## 1.4 Define Functions

In [ ]:
def compute_metrics(cf,
                    ts,
                    cp,
                    obs_col,
                    emb_key,
                    start_res):
    # Define paths
    artifacts_folder_path = '/nfs/team361/sb75/terra-reproducibility/artifacts'
    folder_path = f'{artifacts_folder_path}/hst_corpus_110m/{ts}'
    
    # Load data
    try:
        adata = sc.read_h5ad(f'/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/{ts}/adata_{cp.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    except:
        adata = sc.read_h5ad(f'/nfs/team361/sb75/terra-reproducibility/artifacts/benchmarking/TERRA/{dataset_name}_embedded_clustered_cell_type_{ts}.h5ad')
    adata.uns[emb_key] = {}
    if emb_key == 'neighborhood_emb_layer_12':
        adata.uns[emb_key]['connectivities_key'] = 'neighborhood_connectivities'
        adata.uns[emb_key]['distances_key'] = 'neighborhood_distances'
    elif emb_key == 'cell_emb_layer_12':
        adata.uns[emb_key]['connectivities_key'] = 'cell_connectivities'
        adata.uns[emb_key]['distances_key'] = 'cell_distances'     

    print(adata)

    # Perform clustering until number of ground truth niches is matched
    latent_leiden_resolutions = []
    latent_leiden_resolution = start_res
    i = 0
    n_unique_vals = len(adata.obs[obs_col].value_counts() < 100)
    print(f'Target number of unique values: {n_unique_vals}.')
    n_latent_clusters = 0
    while (n_latent_clusters != n_unique_vals):
        if latent_leiden_resolution in latent_leiden_resolutions:
            latent_leiden_resolution += random.uniform(0.0001, 0.001)
        latent_leiden_resolutions.append(latent_leiden_resolution)
        print(f'Leiden resolution: {latent_leiden_resolution}.')
        latent_cluster_key = f'{emb_key}_emb_leiden_rapids_res{str(latent_leiden_resolution).replace(".", "_")}'
        i += 1
        rsc.tl.leiden(adata,
                      neighbors_key=emb_key,
                      key_added=latent_cluster_key,
                      resolution=latent_leiden_resolution,
                      random_state=0)
    
        n_latent_clusters = len(adata.obs[latent_cluster_key].value_counts() < 100)
        print(f"Number of latent clusters {n_latent_clusters}.")
        if n_latent_clusters == n_unique_vals:
            new_start_res = latent_leiden_resolution
            break
        elif n_latent_clusters < (n_unique_vals - 8):
            latent_leiden_resolution += 0.1
        elif n_latent_clusters < (n_unique_vals - 5):
            latent_leiden_resolution += 0.05
        elif n_latent_clusters < (n_unique_vals - 3):
            latent_leiden_resolution += 0.02
        elif n_latent_clusters < n_unique_vals:
            latent_leiden_resolution += 0.001
        elif n_latent_clusters > (n_unique_vals + 8):
            latent_leiden_resolution -= 0.1
        elif n_latent_clusters > (n_unique_vals + 5):
            latent_leiden_resolution -= 0.05
        elif n_latent_clusters > (n_unique_vals + 3):
            latent_leiden_resolution -= 0.02
        else:
            latent_leiden_resolution -= 0.001
    
    # Compute metrics
    nmis = []
    aris = []
    
    latent_leiden_resolutions = [new_start_res + i * 0.001 for i in range(10)]
    
    for res in latent_leiden_resolutions:
        latent_cluster_key = f'{emb_key.split('_')[0]}_emb_leiden_rapids_res{str(res).replace(".", "_")}'
        
        rsc.tl.leiden(adata,
                      neighbors_key=emb_key,
                      key_added=latent_cluster_key,
                      resolution=res,
                      random_state=0)
    
        if adata.obs[latent_cluster_key].nunique() < adata.obs[obs_col].nunique():
            print(adata.obs[latent_cluster_key].nunique())
            print(adata.obs[obs_col].nunique())
    
        # Compute metrics
        labels = adata.obs[obs_col]
        labels_pred = adata.obs[latent_cluster_key]
        nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
        ari = adjusted_rand_score(labels, labels_pred)
        print(f'NMI is {nmi}. ARI is {ari}.')
        nmis.append(nmi)
        aris.append(ari)
    
    metrics_df = pd.DataFrame({
        'config': [cf] * 20,
        'metric': ['NMI'] * 10 + ['ARI'] * 10,
        'value': nmis + aris
    })
    print(metrics_df)
    print(f'Mean NMI: {np.mean(nmis)}')
    print(f'Mean ARI: {np.mean(aris)}')
    metrics_df.to_csv(f'{folder_path}/{cp.split("-")[1].split(".")[0]}_{dataset_name}_{emb_key.split('_')[0]}_emb_leiden_rapids_avg_metrics.csv',
                      index=False)

# 2. Analysis

## 2.1 Model Size Ablation

### 2.1.1 xhs1000-39b_1p (Xenium Human Skin Eczema) - Niche ID

In [ ]:
# Define dataset config
dataset_name = 'xhs1000-39b_1p'
obsm_key = 'neighborhood_emb_layer_12'
label_col = 'niche_type'

#### 2.1.1.1 Base Model

In [ ]:
config_name = 'gt_base'
load_timestamp = '14112025_074547_785'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.1.2 Small Model

In [ ]:
config_name = 'gt_small'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.1.3 Tiny Model

In [ ]:
config_name = 'gt_tiny'
load_timestamp = '11112025_172307_218'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.1.2 chl59-8b_1p (NanoString CosMx Human Lung NSCLC) - Niche ID

In [ ]:
# Define dataset config
dataset_name = 'chl59-8b_1p'
obsm_key = 'neighborhood_emb_layer_12'
label_col = 'niche'

#### 2.1.2.1 Base Model

In [ ]:
config_name = 'gt_base'
load_timestamp = '14112025_074547_785'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.2.2 Small Model

In [ ]:
config_name = 'gt_small'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.2.3 Tiny Model

In [ ]:
config_name = 'gt_tiny'
load_timestamp = '11112025_172307_218'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.1.3 xhs49-1b_1p (Xenium Human Skin Melanoma) - Niche ID

In [ ]:
# Define dataset config
dataset_name = 'xhs49-1b_1p'
obsm_key = 'neighborhood_emb_layer_12'
label_col = 'niche_name'

#### 2.1.3.1 Base Model

In [ ]:
config_name = 'gt_base'
load_timestamp = '14112025_074547_785'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.3.2 Small Model

In [ ]:
config_name = 'gt_small'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.3.3 Tiny Model

In [ ]:
config_name = 'gt_tiny'
load_timestamp = '11112025_172307_218'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.1.4 chr78-11b_1p (ISS Cartana Human Reproductive Tract) - Niche ID

In [ ]:
# Define dataset config
dataset_name = 'chr78-11b_1p'
obsm_key = 'neighborhood_emb_layer_12'
label_col = 'broad_anatomy'

#### 2.1.4.1 Base Model

In [ ]:
config_name = 'gt_base'
load_timestamp = '14112025_074547_785'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.4.2 Small Model

In [ ]:
config_name = 'gt_small'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.4.3 Tiny Model

In [ ]:
config_name = 'gt_tiny'
load_timestamp = '11112025_172307_218'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.1.5 xhs1000-39b_1p (Xenium Human Skin Eczema) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'xhs1000-39b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'new_annotation'

#### 2.1.5.1 Base Model

In [ ]:
config_name = 'gt_base'
load_timestamp = '14112025_074547_785'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.5.2 Small Model

In [ ]:
config_name = 'gt_small'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.5.3 Tiny Model

In [ ]:
config_name = 'gt_tiny'
load_timestamp = '11112025_172307_218'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.1.6 chl59-8b_1p (NanoString CosMx Human Lung NSCLC) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'chl59-8b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'cell_type'

#### 2.1.6.1 Base Model

In [ ]:
config_name = 'gt_base'
load_timestamp = '14112025_074547_785'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.6.2 Small Model

In [ ]:
config_name = 'gt_small'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.6.3 Tiny Model

In [ ]:
config_name = 'gt_tiny'
load_timestamp = '11112025_172307_218'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.1.7 xhp1016-18b_1p (Xenium Human Pancreas) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'xhp1016-18b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'Annotation_high'

#### 2.1.7.1 Base Model

In [ ]:
config_name = 'gt_base'
load_timestamp = '14112025_074547_785'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.7.2 Small Model

In [ ]:
config_name = 'gt_small'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.7.3 Tiny Model

In [ ]:
config_name = 'gt_tiny'
load_timestamp = '11112025_172307_218'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.1.8 shp75-4b_1p (STARmap Human Placenta) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'shp75-4b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'leiden'

#### 2.1.8.1 Base Model

In [ ]:
config_name = 'gt_base'
load_timestamp = '14112025_074547_785'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.8.2 Small Model

In [ ]:
config_name = 'gt_small'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.8.3 Tiny Model

In [ ]:
config_name = 'gt_tiny'
load_timestamp = '11112025_172307_218'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.1.9 xhk1007-41b_1p (Xenium Human Kidney RCC) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'xhk1007-41b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'level_1_cell_type'

#### 2.1.9.1 Base Model

In [ ]:
config_name = 'gt_base'
load_timestamp = '14112025_074547_785'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.9.2 Small Model

In [ ]:
config_name = 'gt_small'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.9.3 Tiny Model

In [ ]:
config_name = 'gt_tiny'
load_timestamp = '11112025_172307_218'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.1.10 xhb1002-75b_3p (Xenium Human Brain GBM) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'xhb1002-75b_3p'
obsm_key = 'cell_emb_layer_12'
label_col = 'leiden_log1pnorm_res-1'

#### 2.1.10.1 Base Model

In [ ]:
config_name = 'gt_base'
load_timestamp = '14112025_074547_785'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.10.2 Small Model

In [ ]:
config_name = 'gt_small'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.1.10.3 Tiny Model

In [ ]:
config_name = 'gt_tiny'
load_timestamp = '11112025_172307_218'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.1.11 Compare & Visualize

In [ ]:
df_xhs1000_gt_base = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/14112025_074547_785/ep1_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_gt_small = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_gt_small['config'] = 'gt_small'
df_xhs1000_gt_tiny = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/11112025_172307_218/ep3_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')

df_chl59_gt_base = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/14112025_074547_785/ep1_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chl59_gt_small = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chl59_gt_small['config'] = 'gt_small'
df_chl59_gt_tiny = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/11112025_172307_218/ep3_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')

df_xhs49_gt_base = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/14112025_074547_785/ep1_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs49_gt_small = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs49_gt_small['config'] = 'gt_small'
df_xhs49_gt_tiny = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/11112025_172307_218/ep3_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')

df_chr78_gt_base = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/14112025_074547_785/ep1_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chr78_gt_small = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chr78_gt_small['config'] = 'gt_small'
df_chr78_gt_tiny = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/11112025_172307_218/ep3_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')

In [ ]:
df_xhs1000 = pd.concat([df_xhs1000_gt_tiny, df_xhs1000_gt_small, df_xhs1000_gt_base], ignore_index=True)
df_xhs1000['dataset'] = 'Xenium Eczema'
df_chl59 = pd.concat([df_chl59_gt_tiny, df_chl59_gt_small, df_chl59_gt_base], ignore_index=True)
df_chl59['dataset'] = 'NanoString CosMx NSCLC'
df_xhs49 = pd.concat([df_xhs49_gt_tiny, df_xhs49_gt_small, df_xhs49_gt_base], ignore_index=True)
df_xhs49['dataset'] = 'Xenium Melanoma'
df_chr78 = pd.concat([df_chr78_gt_tiny, df_chr78_gt_small, df_chr78_gt_base], ignore_index=True)
df_chr78['dataset'] = 'ISS Cartana Reproductive Tract'

In [ ]:
df_neb = pd.concat([df_xhs49, df_xhs1000, df_chl59, df_chr78], ignore_index=True)

In [ ]:
df_xhs1000_gt_base = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/14112025_074547_785/ep1_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_gt_small = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_gt_small['config'] = 'gt_small'
df_xhs1000_gt_tiny = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/11112025_172307_218/ep3_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')

df_chl59_gt_base = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/14112025_074547_785/ep1_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_chl59_gt_small = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_chl59_gt_small['config'] = 'gt_small'
df_chl59_gt_tiny = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/11112025_172307_218/ep3_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')

df_xhp1016_gt_base = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/14112025_074547_785/ep1_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhp1016_gt_small = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhp1016_gt_small['config'] = 'gt_small'
df_xhp1016_gt_tiny = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/11112025_172307_218/ep3_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')

df_shp75_gt_base = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/14112025_074547_785/ep1_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_shp75_gt_small = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_shp75_gt_small['config'] = 'gt_small'
df_shp75_gt_tiny = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/11112025_172307_218/ep3_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')

In [ ]:
df_xhs1000 = pd.concat([df_xhs1000_gt_tiny, df_xhs1000_gt_small, df_xhs1000_gt_base], ignore_index=True)
df_xhs1000['dataset'] = 'Xenium Eczema'
df_chl59 = pd.concat([df_chl59_gt_tiny, df_chl59_gt_small, df_chl59_gt_base], ignore_index=True)
df_chl59['dataset'] = 'NanoString CosMx NSCLC'
df_xhp1016 = pd.concat([df_xhp1016_gt_tiny, df_xhp1016_gt_small, df_xhp1016_gt_base], ignore_index=True)
df_xhp1016['dataset'] = 'Xenium Developing Pancreas'
df_shp75 = pd.concat([df_shp75_gt_tiny, df_shp75_gt_small, df_shp75_gt_base], ignore_index=True)
df_shp75['dataset'] = 'STARmap Healthy Placenta'

In [ ]:
df_cell = pd.concat([df_xhs1000, df_chl59, df_xhp1016, df_shp75], ignore_index=True)

In [ ]:
#df_xhk1007_gt_base = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/14112025_074547_785/ep1_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhk1007_gt_small = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhk1007_gt_small['config'] = 'gt_small'
#df_xhk1007_gt_tiny = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/11112025_172307_218/ep3_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')

#df_xhb1002_gt_base = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/14112025_074547_785/ep1_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhb1002_gt_small = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhb1002_gt_small['config'] = 'gt_small'
#df_xhb1002_gt_tiny = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/11112025_172307_218/ep3_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')

In [ ]:
#df_xhk1007 = pd.concat([df_xhk1007_gt_tiny, df_xhk1007_gt_small, df_xhk1007_gt_base], ignore_index=True)
#df_xhk1007['dataset'] = 'Xenium Kidney RCC'
#df_xhb1002 = pd.concat([df_xhb1002_gt_tiny, df_xhb1002_gt_small, df_xhb1002_gt_base], ignore_index=True)
#df_xhb1002['dataset'] = 'Xenium Brain GBM'

In [ ]:
#df_cell = pd.concat([df_xhk1007, df_xhb1002], ignore_index=True)

In [ ]:
# --- Filter only NMI metric ---
df_nmi = df_neb[df_neb["metric"] == "NMI"]

# --- Compute mean and std per dataset × config ---
stats = (
    df_nmi.groupby(["dataset", "config"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# --- Enforce dataset order ---
desired_order = [
    "Xenium Eczema",
    "NanoString CosMx NSCLC",
    "Xenium Melanoma",
    "ISS Cartana Reproductive Tract",
]
stats["dataset"] = pd.Categorical(stats["dataset"], categories=desired_order, ordered=True)
stats = stats.sort_values("dataset").reset_index(drop=True)

# --- Configuration ---
configs = ["gt_tiny", "gt_small", "gt_base"]
config_labels = {
    "gt_tiny": "GT Tiny (10M)",
    "gt_small": "GT Small (30M)",
    "gt_base": "GT Base (103M)",
}
colors = ["#4477AA", "#DDCC77", "#CC6677"]

datasets = stats["dataset"].unique()
bar_width = 0.25
x = range(len(datasets))

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

# --- Plot bars for each configuration ---
for i, cfg in enumerate(configs):
    subset = stats[stats["config"] == cfg]
    ax.bar(
        [xi + (i - 1) * bar_width for xi in x],
        subset["mean"],
        yerr=subset["std"],
        width=bar_width,
        label=config_labels[cfg],
        color=colors[i],
        edgecolor="black",
        linewidth=0.8,
        capsize=3,
    )

# --- Wrap labels after the first word ---
def wrap_after_first_word(label):
    parts = label.split(" ", 1)
    return "\n".join(parts) if len(parts) > 1 else label

wrapped_labels = [wrap_after_first_word(ds) for ds in datasets]

ax.set_xticks(x)
ax.set_xticklabels(wrapped_labels, fontsize=12, ha="center")

# --- Aesthetics ---
ax.set_ylabel("NMI", fontsize=12)
ax.set_ylim(0, 1.0)

# Boxed legend (clean, professional look)
legend = ax.legend(
    frameon=True,
    fontsize=12,
    loc="upper right",
    framealpha=0.95,
    fancybox=True,
    edgecolor="black",
    title="Model size",
    title_fontsize=12,
)
legend.get_frame().set_linewidth(0.6)

# Clean spines and grid
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="0.9", linestyle="-", linewidth=0.5)
ax.set_axisbelow(True)

save_path = os.path.join(sc.settings.figdir, "model_size_niche_nmi_comparison.svg")
plt.savefig(save_path, bbox_inches="tight", format="svg")
print(f"✅ Figure saved to: {save_path}")

plt.tight_layout()
plt.show()

In [ ]:
# --- Filter only ARI metric ---
df_nmi = df_neb[df_neb["metric"] == "ARI"]

# --- Compute mean and std per dataset × config ---
stats = (
    df_nmi.groupby(["dataset", "config"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# --- Enforce dataset order ---
desired_order = [
    "Xenium Eczema",
    "NanoString CosMx NSCLC",
    "Xenium Melanoma",
    "ISS Cartana Reproductive Tract",
]
stats["dataset"] = pd.Categorical(stats["dataset"], categories=desired_order, ordered=True)
stats = stats.sort_values("dataset").reset_index(drop=True)

# --- Configuration ---
configs = ["gt_tiny", "gt_small", "gt_base"]
config_labels = {
    "gt_tiny": "GT Tiny (10M)",
    "gt_small": "GT Small (30M)",
    "gt_base": "GT Base (103M)",
}
colors = ["#4477AA", "#DDCC77", "#CC6677"]

datasets = stats["dataset"].unique()
bar_width = 0.25
x = range(len(datasets))

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

# --- Plot bars for each configuration ---
for i, cfg in enumerate(configs):
    subset = stats[stats["config"] == cfg]
    ax.bar(
        [xi + (i - 1) * bar_width for xi in x],
        subset["mean"],
        yerr=subset["std"],
        width=bar_width,
        label=config_labels[cfg],
        color=colors[i],
        edgecolor="black",
        linewidth=0.8,
        capsize=3,
    )

# --- Wrap labels after the first word ---
def wrap_after_first_word(label):
    parts = label.split(" ", 1)
    return "\n".join(parts) if len(parts) > 1 else label

wrapped_labels = [wrap_after_first_word(ds) for ds in datasets]

ax.set_xticks(x)
ax.set_xticklabels(wrapped_labels, fontsize=12, ha="center")

# --- Aesthetics ---
ax.set_ylabel("ARI", fontsize=12)
ax.set_ylim(0, 1.0)

# Boxed legend (clean, professional look)
legend = ax.legend(
    frameon=True,
    fontsize=12,
    loc="upper right",
    framealpha=0.95,
    fancybox=True,
    edgecolor="black",
    title="Model size",
    title_fontsize=12,
)
legend.get_frame().set_linewidth(0.6)

# Clean spines and grid
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="0.9", linestyle="-", linewidth=0.5)
ax.set_axisbelow(True)

save_path = os.path.join(sc.settings.figdir, "model_size_niche_ari_comparison.svg")
plt.savefig(save_path, bbox_inches="tight", format="svg")
print(f"✅ Figure saved to: {save_path}")

plt.tight_layout()
plt.show()

In [ ]:
# --- Filter only NMI metric ---
df_nmi = df_cell[df_cell["metric"] == "NMI"]

# --- Compute mean and std per dataset × config ---
stats = (
    df_nmi.groupby(["dataset", "config"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# --- Enforce dataset order ---
desired_order = [
    'Xenium Eczema',
    'NanoString CosMx NSCLC',
    'Xenium Developing Pancreas',
    'STARmap Healthy Placenta'
]
stats["dataset"] = pd.Categorical(stats["dataset"], categories=desired_order, ordered=True)
stats = stats.sort_values("dataset").reset_index(drop=True)

# --- Configuration ---
configs = ["gt_tiny", "gt_small", "gt_base"]
config_labels = {
    "gt_tiny": "GT Tiny (10M)",
    "gt_small": "GT Small (30M)",
    "gt_base": "GT Base (103M)",
}
colors = ["#4477AA", "#DDCC77", "#CC6677"]

datasets = stats["dataset"].unique()
bar_width = 0.25
x = range(len(datasets))

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

# --- Plot bars for each configuration ---
for i, cfg in enumerate(configs):
    subset = stats[stats["config"] == cfg]
    ax.bar(
        [xi + (i - 1) * bar_width for xi in x],
        subset["mean"],
        yerr=subset["std"],
        width=bar_width,
        label=config_labels[cfg],
        color=colors[i],
        edgecolor="black",
        linewidth=0.8,
        capsize=3,
    )

# --- Wrap labels after the first word ---
def wrap_after_first_word(label):
    parts = label.split(" ", 1)
    return "\n".join(parts) if len(parts) > 1 else label

wrapped_labels = [wrap_after_first_word(ds) for ds in datasets]

ax.set_xticks(x)
ax.set_xticklabels(wrapped_labels, fontsize=12, ha="center")

# --- Aesthetics ---
ax.set_ylabel("NMI", fontsize=12)
ax.set_ylim(0, 1.0)

# Boxed legend (clean, professional look)
legend = ax.legend(
    frameon=True,
    fontsize=12,
    loc="upper right",
    framealpha=0.95,
    fancybox=True,
    edgecolor="black",
    title="Model size",
    title_fontsize=12,
)
legend.get_frame().set_linewidth(0.6)

# Clean spines and grid
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="0.9", linestyle="-", linewidth=0.5)
ax.set_axisbelow(True)

save_path = os.path.join(sc.settings.figdir, "model_size_cell_type_nmi_comparison.svg")
plt.savefig(save_path, bbox_inches="tight", format="svg")
print(f"✅ Figure saved to: {save_path}")

plt.tight_layout()
plt.show()

In [ ]:
# --- Filter only ARI metric ---
df_nmi = df_cell[df_cell["metric"] == "ARI"]

# --- Compute mean and std per dataset × config ---
stats = (
    df_nmi.groupby(["dataset", "config"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# --- Enforce dataset order ---
desired_order = [
    'Xenium Eczema',
    'NanoString CosMx NSCLC',
    'Xenium Developing Pancreas',
    'STARmap Healthy Placenta'
]
stats["dataset"] = pd.Categorical(stats["dataset"], categories=desired_order, ordered=True)
stats = stats.sort_values("dataset").reset_index(drop=True)

# --- Configuration ---
configs = ["gt_tiny", "gt_small", "gt_base"]
config_labels = {
    "gt_tiny": "GT Tiny (10M)",
    "gt_small": "GT Small (30M)",
    "gt_base": "GT Base (103M)",
}
colors = ["#4477AA", "#DDCC77", "#CC6677"]

datasets = stats["dataset"].unique()
bar_width = 0.25
x = range(len(datasets))

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

# --- Plot bars for each configuration ---
for i, cfg in enumerate(configs):
    subset = stats[stats["config"] == cfg]
    ax.bar(
        [xi + (i - 1) * bar_width for xi in x],
        subset["mean"],
        yerr=subset["std"],
        width=bar_width,
        label=config_labels[cfg],
        color=colors[i],
        edgecolor="black",
        linewidth=0.8,
        capsize=3,
    )

# --- Wrap labels after the first word ---
def wrap_after_first_word(label):
    parts = label.split(" ", 1)
    return "\n".join(parts) if len(parts) > 1 else label

wrapped_labels = [wrap_after_first_word(ds) for ds in datasets]

ax.set_xticks(x)
ax.set_xticklabels(wrapped_labels, fontsize=12, ha="center")

# --- Aesthetics ---
ax.set_ylabel("ARI", fontsize=12)
ax.set_ylim(0, 1.0)

# Boxed legend (clean, professional look)
legend = ax.legend(
    frameon=True,
    fontsize=12,
    loc="upper right",
    framealpha=0.95,
    fancybox=True,
    edgecolor="black",
    title="Model size",
    title_fontsize=12,
)
legend.get_frame().set_linewidth(0.6)

# Clean spines and grid
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="0.9", linestyle="-", linewidth=0.5)
ax.set_axisbelow(True)

save_path = os.path.join(sc.settings.figdir, "model_size_cell_type_ari_comparison.svg")
plt.savefig(save_path, bbox_inches="tight", format="svg")
print(f"✅ Figure saved to: {save_path}")

plt.tight_layout()
plt.show()

## 2.2 Dataset Size Ablation

### 2.2.1 xhs1000-39b_1p (Xenium Human Skin Eczema) - Niche ID

In [ ]:
# Define dataset config
dataset_name = 'xhs1000-39b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'new_annotation'

#### 2.2.1.1 1 Percent Sample

In [ ]:
config_name = '1_pct_sample'
load_timestamp = '12112025_132402_667'
read_checkpoint = 'terra-ep199.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.1.2 10 Percent Sample

In [ ]:
config_name = '10_pct_sample'
load_timestamp = '08112025_183353_702'
read_checkpoint = 'terra-ep19.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.1.3 25 Percent Sample

In [ ]:
config_name = '25_pct_sample'
load_timestamp = '10112025_185828_463'
read_checkpoint = 'terra-ep7.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.1.4 50 Percent Sample

In [ ]:
config_name = '50_pct_sample'
load_timestamp = '09112025_190730_770'
read_checkpoint = 'terra-ep5.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.1.5 100 Percent Sample

In [ ]:
config_name = '100_pct_sample'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.2.2 chl59-8b_1p (NanoString CosMx Human Lung NSCLC) - Niche ID

In [ ]:
# Define dataset config
dataset_name = 'chl59-8b_1p'
obsm_key = 'neighborhood_emb_layer_12'
label_col = 'niche'

#### 2.2.2.1 1 Percent Sample

In [ ]:
config_name = '1_pct_sample'
load_timestamp = '12112025_132402_667'
read_checkpoint = 'terra-ep199.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.2.2 10 Percent Sample

In [ ]:
config_name = '10_pct_sample'
load_timestamp = '08112025_183353_702'
read_checkpoint = 'terra-ep19.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.2.3 25 Percent Sample

In [ ]:
config_name = '25_pct_sample'
load_timestamp = '10112025_185828_463'
read_checkpoint = 'terra-ep7.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.2.4 50 Percent Sample

In [ ]:
config_name = '50_pct_sample'
load_timestamp = '09112025_190730_770'
read_checkpoint = 'terra-ep5.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.2.5 100 Percent Sample

In [ ]:
config_name = '100_pct_sample'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.2.3 xhs49-1b_1p (Xenium Human Skin Melanoma) - Niche ID

In [ ]:
# Define dataset config
dataset_name = 'xhs49-1b_1p'
obsm_key = 'neighborhood_emb_layer_12'
label_col = 'niche_name'

#### 2.2.3.1 1 Percent Sample

In [ ]:
config_name = '1_pct_sample'
load_timestamp = '12112025_132402_667'
read_checkpoint = 'terra-ep199.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.3.2 10 Percent Sample

In [ ]:
config_name = '10_pct_sample'
load_timestamp = '08112025_183353_702'
read_checkpoint = 'terra-ep19.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.3.3 25 Percent Sample

In [ ]:
config_name = '25_pct_sample'
load_timestamp = '10112025_185828_463'
read_checkpoint = 'terra-ep7.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.3.4 50 Percent Sample

In [ ]:
config_name = '50_pct_sample'
load_timestamp = '09112025_190730_770'
read_checkpoint = 'terra-ep5.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.3.5 100 Percent Sample

In [ ]:
config_name = '100_pct_sample'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.2.4 chr78-11b_1p (ISS Cartana Human Reproductive Tract) - Niche ID

In [ ]:
# Define dataset config
dataset_name = 'chr78-11b_1p'
obsm_key = 'neighborhood_emb_layer_12'
label_col = 'broad_anatomy'

#### 2.2.4.1 1 Percent Sample

In [ ]:
config_name = '1_pct_sample'
load_timestamp = '12112025_132402_667'
read_checkpoint = 'terra-ep199.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.4.2 10 Percent Sample

In [ ]:
config_name = '10_pct_sample'
load_timestamp = '08112025_183353_702'
read_checkpoint = 'terra-ep19.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.4.3 25 Percent Sample

In [ ]:
config_name = '25_pct_sample'
load_timestamp = '10112025_185828_463'
read_checkpoint = 'terra-ep7.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.4.4 50 Percent Sample

In [ ]:
config_name = '50_pct_sample'
load_timestamp = '09112025_190730_770'
read_checkpoint = 'terra-ep5.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.4.5 100 Percent Sample

In [ ]:
config_name = '100_pct_sample'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.2.5 xhs1000-39b_1p (Xenium Human Skin Eczema) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'xhs1000-39b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'new_annotation'

#### 2.2.5.1 1 Percent Sample

In [ ]:
config_name = '1_pct_sample'
load_timestamp = '12112025_132402_667'
read_checkpoint = 'terra-ep199.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.5.2 10 Percent Sample

In [ ]:
config_name = '10_pct_sample'
load_timestamp = '08112025_183353_702'
read_checkpoint = 'terra-ep19.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.5.3 25 Percent Sample

In [ ]:
config_name = '25_pct_sample'
load_timestamp = '10112025_185828_463'
read_checkpoint = 'terra-ep7.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.5.4 50 Percent Sample

In [ ]:
config_name = '50_pct_sample'
load_timestamp = '09112025_190730_770'
read_checkpoint = 'terra-ep5.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.5.5 100 Percent Sample

In [ ]:
config_name = '100_pct_sample'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.2.6 chl59-8b_1p (NanoString CosMx Human Lung NSCLC) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'chl59-8b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'cell_type'

#### 2.2.6.1 1 Percent Sample

In [ ]:
config_name = '1_pct_sample'
load_timestamp = '12112025_132402_667'
read_checkpoint = 'terra-ep199.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.6.2 10 Percent Sample

In [ ]:
config_name = '10_pct_sample'
load_timestamp = '08112025_183353_702'
read_checkpoint = 'terra-ep19.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.6.3 25 Percent Sample

In [ ]:
config_name = '25_pct_sample'
load_timestamp = '10112025_185828_463'
read_checkpoint = 'terra-ep7.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.6.4 50 Percent Sample

In [ ]:
config_name = '50_pct_sample'
load_timestamp = '09112025_190730_770'
read_checkpoint = 'terra-ep5.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.6.5 100 Percent Sample

In [ ]:
config_name = '100_pct_sample'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.2.7 xhp1016-18b_1p (Xenium Human Pancreas) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'xhp1016-18b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'Annotation_high'

#### 2.2.7.1 1 Percent Sample

In [ ]:
config_name = '1_pct_sample'
load_timestamp = '12112025_132402_667'
read_checkpoint = 'terra-ep199.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.7.2 10 Percent Sample

In [ ]:
config_name = '10_pct_sample'
load_timestamp = '08112025_183353_702'
read_checkpoint = 'terra-ep19.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.7.3 25 Percent Sample

In [ ]:
config_name = '25_pct_sample'
load_timestamp = '10112025_185828_463'
read_checkpoint = 'terra-ep7.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.7.4 50 Percent Sample

In [ ]:
config_name = '50_pct_sample'
load_timestamp = '09112025_190730_770'
read_checkpoint = 'terra-ep5.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.7.5 100 Percent Sample

In [ ]:
config_name = '100_pct_sample'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.2.8 shp75-4b_1p (STARmap Human Placenta) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'shp75-4b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'leiden'

#### 2.2.8.1 1 Percent Sample

In [ ]:
config_name = '1_pct_sample'
load_timestamp = '12112025_132402_667'
read_checkpoint = 'terra-ep199.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.8.2 10 Percent Sample

In [ ]:
config_name = '10_pct_sample'
load_timestamp = '08112025_183353_702'
read_checkpoint = 'terra-ep19.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.8.3 25 Percent Sample

In [ ]:
config_name = '25_pct_sample'
load_timestamp = '10112025_185828_463'
read_checkpoint = 'terra-ep7.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.8.4 50 Percent Sample

In [ ]:
config_name = '50_pct_sample'
load_timestamp = '09112025_190730_770'
read_checkpoint = 'terra-ep5.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.8.5 100 Percent Sample

In [ ]:
config_name = '100_pct_sample'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.2.9 xhk1007-41b_1p (Xenium Human Kidney RCC) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'xhk1007-41b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'level_1_cell_type'

#### 2.2.9.1 1 Percent Sample

In [ ]:
config_name = '1_pct_sample'
load_timestamp = '12112025_132402_667'
read_checkpoint = 'terra-ep199.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.9.2 10 Percent Sample

In [ ]:
config_name = '10_pct_sample'
load_timestamp = '08112025_183353_702'
read_checkpoint = 'terra-ep19.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.9.3 25 Percent Sample

In [ ]:
config_name = '25_pct_sample'
load_timestamp = '10112025_185828_463'
read_checkpoint = 'terra-ep7.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.9.4 50 Percent Sample

In [ ]:
config_name = '50_pct_sample'
load_timestamp = '09112025_190730_770'
read_checkpoint = 'terra-ep5.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.9.5 100 Percent Sample

In [ ]:
config_name = '100_pct_sample'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.2.10 xhb1002-75b_3p (Xenium Human Brain GBM) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'xhb1002-75b_3p'
obsm_key = 'cell_emb_layer_12'
label_col = 'leiden_log1pnorm_res-1'

#### 2.2.10.1 1 Percent Sample

In [ ]:
config_name = '1_pct_sample'
load_timestamp = '12112025_132402_667'
read_checkpoint = 'terra-ep199.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.10.2 10 Percent Sample

In [ ]:
config_name = '10_pct_sample'
load_timestamp = '08112025_183353_702'
read_checkpoint = 'terra-ep19.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.10.3 25 Percent Sample

In [ ]:
config_name = '25_pct_sample'
load_timestamp = '10112025_185828_463'
read_checkpoint = 'terra-ep7.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.10.4 50 Percent Sample

In [ ]:
config_name = '50_pct_sample'
load_timestamp = '09112025_190730_770'
read_checkpoint = 'terra-ep5.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.2.10.5 100 Percent Sample

In [ ]:
config_name = '100_pct_sample'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.2.11 Compare & Visualize

In [ ]:
df_xhs1000_1_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/12112025_132402_667/ep199_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_10_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/08112025_183353_702/ep19_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_25_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/10112025_185828_463/ep7_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_50_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/09112025_190730_770/ep5_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_100_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_100_pct_sample['config'] = '100_pct_sample'

df_chl59_1_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/12112025_132402_667/ep199_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chl59_10_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/08112025_183353_702/ep19_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chl59_25_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/10112025_185828_463/ep7_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chl59_50_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/09112025_190730_770/ep5_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chl59_100_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chl59_100_pct_sample['config'] = '100_pct_sample'

df_xhs49_1_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/12112025_132402_667/ep199_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs49_10_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/08112025_183353_702/ep19_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs49_25_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/10112025_185828_463/ep7_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs49_50_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/09112025_190730_770/ep5_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs49_100_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs49_100_pct_sample['config'] = '100_pct_sample'

df_chr78_1_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/12112025_132402_667/ep199_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chr78_10_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/08112025_183353_702/ep19_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chr78_25_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/10112025_185828_463/ep7_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chr78_50_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/09112025_190730_770/ep5_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chr78_100_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chr78_100_pct_sample['config'] = '100_pct_sample'

In [ ]:
df_xhs1000 = pd.concat([df_xhs1000_100_pct_sample, df_xhs1000_50_pct_sample, df_xhs1000_25_pct_sample, df_xhs1000_10_pct_sample, df_xhs1000_1_pct_sample], ignore_index=True)
df_xhs1000['dataset'] = 'Xenium Eczema'
df_chl59 = pd.concat([df_chl59_100_pct_sample, df_chl59_50_pct_sample, df_chl59_25_pct_sample, df_chl59_10_pct_sample, df_chl59_1_pct_sample], ignore_index=True)
df_chl59['dataset'] = 'NanoString CosMx NSCLC'
df_xhs49 = pd.concat([df_xhs49_100_pct_sample, df_xhs49_50_pct_sample, df_xhs49_25_pct_sample, df_xhs49_10_pct_sample, df_xhs49_1_pct_sample], ignore_index=True)
df_xhs49['dataset'] = 'Xenium Melanoma'
df_chr78 = pd.concat([df_chr78_100_pct_sample, df_chr78_50_pct_sample, df_chr78_25_pct_sample, df_chr78_10_pct_sample, df_chr78_1_pct_sample], ignore_index=True)
df_chr78['dataset'] = 'ISS Cartana Reproductive Tract'

In [ ]:
df_neb = pd.concat([df_xhs1000, df_chl59, df_xhs49, df_chr78], ignore_index=True)

In [ ]:
df_xhs1000_1_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/12112025_132402_667/ep199_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_10_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/08112025_183353_702/ep19_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_25_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/10112025_185828_463/ep7_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_50_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/09112025_190730_770/ep5_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_100_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_100_pct_sample['config'] = '100_pct_sample'

df_chl59_1_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/12112025_132402_667/ep199_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_chl59_10_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/08112025_183353_702/ep19_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_chl59_25_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/10112025_185828_463/ep7_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_chl59_50_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/09112025_190730_770/ep5_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_chl59_100_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_chl59_100_pct_sample['config'] = '100_pct_sample'

df_xhp1016_1_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/12112025_132402_667/ep199_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhp1016_10_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/08112025_183353_702/ep19_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhp1016_25_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/10112025_185828_463/ep7_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhp1016_50_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/09112025_190730_770/ep5_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhp1016_100_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhp1016_100_pct_sample['config'] = '100_pct_sample'

df_shp75_1_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/12112025_132402_667/ep199_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_shp75_10_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/08112025_183353_702/ep19_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_shp75_25_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/10112025_185828_463/ep7_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_shp75_50_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/09112025_190730_770/ep5_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_shp75_100_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_shp75_100_pct_sample['config'] = '100_pct_sample'

In [ ]:
df_xhs1000 = pd.concat([df_xhs1000_100_pct_sample, df_xhs1000_50_pct_sample, df_xhs1000_25_pct_sample, df_xhs1000_10_pct_sample, df_xhs1000_1_pct_sample], ignore_index=True)
df_xhs1000['dataset'] = 'Xenium Eczema'
df_chl59 = pd.concat([df_chl59_100_pct_sample, df_chl59_50_pct_sample, df_chl59_25_pct_sample, df_chl59_10_pct_sample, df_chl59_1_pct_sample], ignore_index=True)
df_chl59['dataset'] = 'NanoString CosMx NSCLC'
df_xhp1016 = pd.concat([df_xhp1016_100_pct_sample, df_xhp1016_50_pct_sample, df_xhp1016_25_pct_sample, df_xhp1016_10_pct_sample, df_xhp1016_1_pct_sample], ignore_index=True)
df_xhp1016['dataset'] = 'Xenium Developing Pancreas'
df_shp75 = pd.concat([df_shp75_100_pct_sample, df_shp75_50_pct_sample, df_shp75_25_pct_sample, df_shp75_10_pct_sample, df_shp75_1_pct_sample], ignore_index=True)
df_shp75['dataset'] = 'STARmap Healthy Placenta'

In [ ]:
df_cell = pd.concat([df_xhs1000, df_chl59, df_xhp1016, df_shp75], ignore_index=True)

In [ ]:
#df_xhk1007_1_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/12112025_132402_667/ep199_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhk1007_10_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/08112025_183353_702/ep19_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhk1007_25_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/10112025_185828_463/ep7_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhk1007_50_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/09112025_190730_770/ep5_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhk1007_100_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhk1007_100_pct_sample['config'] = '100_pct_sample'

#df_xhb1002_1_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/12112025_132402_667/ep199_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhb1002_10_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/08112025_183353_702/ep19_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhb1002_25_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/10112025_185828_463/ep7_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhb1002_50_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/09112025_190730_770/ep5_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhb1002_100_pct_sample = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhb1002_100_pct_sample['config'] = '100_pct_sample'

In [ ]:
#df_xhk1007 = pd.concat([df_xhk1007_100_pct_sample, df_xhk1007_50_pct_sample, df_xhk1007_25_pct_sample, df_xhk1007_10_pct_sample, df_xhk1007_1_pct_sample], ignore_index=True)
#df_xhk1007['dataset'] = 'Xenium Kidney RCC'
#df_xhb1002 = pd.concat([df_xhb1002_100_pct_sample, df_xhb1002_50_pct_sample, df_xhb1002_25_pct_sample, df_xhb1002_10_pct_sample, df_xhb1002_1_pct_sample], ignore_index=True)
#df_xhb1002['dataset'] = 'Xenium Brain GBM'

In [ ]:
#df_cell = pd.concat([df_xhk1007, df_xhb1002], ignore_index=True)

In [ ]:
# --- Filter only NMI metric ---
df_nmi = df_neb[df_neb["metric"] == "NMI"]

# --- Compute mean and std per dataset × config ---
stats = (
    df_nmi.groupby(["dataset", "config"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# --- Enforce dataset order ---
desired_order = [
    "Xenium Eczema",
    "NanoString CosMx NSCLC",
    "Xenium Melanoma",
    "ISS Cartana Reproductive Tract",
]
stats["dataset"] = pd.Categorical(stats["dataset"], categories=desired_order, ordered=True)
stats = stats.sort_values("dataset").reset_index(drop=True)

# --- Configurations ---
configs = sorted(stats["config"].unique(), key=lambda x: float(x.split("_")[0]))  # e.g. "1_pct_sample" → 1.0
config_labels = [cfg.replace("_", " ").replace("pct", "%").capitalize() for cfg in configs]

# --- Continuous colormap (light→dark gradient) ---
cmap = cm.get_cmap("viridis", len(configs))
colors = [cmap(i) for i in range(len(configs))]

datasets = stats["dataset"].unique()
bar_width = 0.8 / len(configs)  # keep bars compact
x = range(len(datasets))

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

# --- Plot bars for each configuration ---
for i, cfg in enumerate(configs):
    subset = stats[stats["config"] == cfg]
    ax.bar(
        [xi + (i - len(configs)/2) * bar_width for xi in x],
        subset["mean"],
        yerr=subset["std"],
        width=bar_width,
        label=config_labels[i],
        color=colors[i],
        edgecolor="black",
        linewidth=0.5,
        capsize=2,
    )

# --- Wrap labels after the first word ---
def wrap_after_first_word(label):
    label = str(label)
    parts = label.split(" ", 1)
    return "\n".join(parts) if len(parts) > 1 else label

wrapped_labels = [wrap_after_first_word(ds) for ds in datasets]
ax.set_xticks(x)
ax.set_xticklabels(wrapped_labels, fontsize=12, ha="center")

# --- Aesthetics ---
ax.set_ylabel("NMI", fontsize=12)
ax.set_ylim(0, 1.0)

# --- Legend with colormap box ---
legend = ax.legend(
    frameon=True,
    fontsize=12,
    loc="upper right",
    framealpha=0.95,
    fancybox=True,
    edgecolor="black",
    title="Dataset size",
    title_fontsize=12,
)
legend.get_frame().set_linewidth(0.6)

# --- Clean spines and grid ---
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="0.9", linestyle="-", linewidth=0.5)
ax.set_axisbelow(True)

# --- Save figure ---
save_path = os.path.join(sc.settings.figdir, "dataset_size_niche_nmi_comparison.svg")
plt.savefig(save_path, bbox_inches="tight", format="svg")
print(f"✅ Figure saved to: {save_path}")

plt.tight_layout()
plt.show()

In [ ]:
# --- Filter only ARI metric ---
df_ari = df_neb[df_neb["metric"] == "ARI"]

# --- Compute mean and std per dataset × config ---
stats = (
    df_ari.groupby(["dataset", "config"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# --- Enforce dataset order ---
desired_order = [
    "Xenium Eczema",
    "NanoString CosMx NSCLC",
    "Xenium Melanoma",
    "ISS Cartana Reproductive Tract",
]
stats["dataset"] = pd.Categorical(stats["dataset"], categories=desired_order, ordered=True)
stats = stats.sort_values("dataset").reset_index(drop=True)

# --- Configurations ---
configs = sorted(stats["config"].unique(), key=lambda x: float(x.split("_")[0]))  # e.g. "1_pct_sample" → 1.0
config_labels = [cfg.replace("_", " ").replace("pct", "%").capitalize() for cfg in configs]

# --- Continuous colormap (light→dark gradient) ---
cmap = cm.get_cmap("viridis", len(configs))
colors = [cmap(i) for i in range(len(configs))]

datasets = stats["dataset"].unique()
bar_width = 0.8 / len(configs)  # keep bars compact
x = range(len(datasets))

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

# --- Plot bars for each configuration ---
for i, cfg in enumerate(configs):
    subset = stats[stats["config"] == cfg]
    ax.bar(
        [xi + (i - len(configs)/2) * bar_width for xi in x],
        subset["mean"],
        yerr=subset["std"],
        width=bar_width,
        label=config_labels[i],
        color=colors[i],
        edgecolor="black",
        linewidth=0.5,
        capsize=2,
    )

# --- Wrap labels after the first word ---
def wrap_after_first_word(label):
    label = str(label)
    parts = label.split(" ", 1)
    return "\n".join(parts) if len(parts) > 1 else label

wrapped_labels = [wrap_after_first_word(ds) for ds in datasets]
ax.set_xticks(x)
ax.set_xticklabels(wrapped_labels, fontsize=12, ha="center")

# --- Aesthetics ---
ax.set_ylabel("ARI", fontsize=12)
ax.set_ylim(0, 1.0)

# --- Legend with colormap box ---
legend = ax.legend(
    frameon=True,
    fontsize=12,
    loc="upper right",
    framealpha=0.95,
    fancybox=True,
    edgecolor="black",
    title="Dataset size",
    title_fontsize=12,
)
legend.get_frame().set_linewidth(0.6)

# --- Clean spines and grid ---
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="0.9", linestyle="-", linewidth=0.5)
ax.set_axisbelow(True)

# --- Save figure ---
save_path = os.path.join(sc.settings.figdir, "dataset_size_niche_ari_comparison.svg")
plt.savefig(save_path, bbox_inches="tight", format="svg")
print(f"✅ Figure saved to: {save_path}")

plt.tight_layout()
plt.show()

In [ ]:
# --- Filter only NMI metric ---
df_nmi = df_cell[df_cell["metric"] == "NMI"]

# --- Compute mean and std per dataset × config ---
stats = (
    df_nmi.groupby(["dataset", "config"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# --- Enforce dataset order ---
desired_order = [
    'Xenium Eczema',
    'NanoString CosMx NSCLC',
    'Xenium Developing Pancreas',
    'STARmap Healthy Placenta'
]
stats["dataset"] = pd.Categorical(stats["dataset"], categories=desired_order, ordered=True)
stats = stats.sort_values("dataset").reset_index(drop=True)

# --- Configurations ---
configs = sorted(stats["config"].unique(), key=lambda x: float(x.split("_")[0]))  # e.g. "1_pct_sample" → 1.0
config_labels = [cfg.replace("_", " ").replace("pct", "%").capitalize() for cfg in configs]

# --- Continuous colormap (light→dark gradient) ---
cmap = cm.get_cmap("viridis", len(configs))
colors = [cmap(i) for i in range(len(configs))]

datasets = stats["dataset"].unique()
bar_width = 0.8 / len(configs)  # keep bars compact
x = range(len(datasets))

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

# --- Plot bars for each configuration ---
for i, cfg in enumerate(configs):
    subset = stats[stats["config"] == cfg]
    ax.bar(
        [xi + (i - len(configs)/2) * bar_width for xi in x],
        subset["mean"],
        yerr=subset["std"],
        width=bar_width,
        label=config_labels[i],
        color=colors[i],
        edgecolor="black",
        linewidth=0.5,
        capsize=2,
    )

# --- Wrap labels after the first word ---
def wrap_after_first_word(label):
    label = str(label)
    parts = label.split(" ", 1)
    return "\n".join(parts) if len(parts) > 1 else label

wrapped_labels = [wrap_after_first_word(ds) for ds in datasets]
ax.set_xticks(x)
ax.set_xticklabels(wrapped_labels, fontsize=12, ha="center")

# --- Aesthetics ---
ax.set_ylabel("NMI", fontsize=12)
ax.set_ylim(0, 1.0)

# --- Legend with colormap box ---
legend = ax.legend(
    frameon=True,
    fontsize=12,
    loc="upper right",
    framealpha=0.95,
    fancybox=True,
    edgecolor="black",
    title="Dataset size",
    title_fontsize=12,
)
legend.get_frame().set_linewidth(0.6)

# --- Clean spines and grid ---
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="0.9", linestyle="-", linewidth=0.5)
ax.set_axisbelow(True)

# --- Save figure ---
save_path = os.path.join(sc.settings.figdir, "dataset_size_cell_type_nmi_comparison.svg")
plt.savefig(save_path, bbox_inches="tight", format="svg")
print(f"✅ Figure saved to: {save_path}")

plt.tight_layout()
plt.show()

In [ ]:
# --- Filter only ARI metric ---
df_ari = df_cell[df_cell["metric"] == "ARI"]

# --- Compute mean and std per dataset × config ---
stats = (
    df_ari.groupby(["dataset", "config"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# --- Enforce dataset order ---
desired_order = [
    'Xenium Eczema',
    'NanoString CosMx NSCLC',
    'Xenium Developing Pancreas',
    'STARmap Healthy Placenta'
]
stats["dataset"] = pd.Categorical(stats["dataset"], categories=desired_order, ordered=True)
stats = stats.sort_values("dataset").reset_index(drop=True)

# --- Configurations ---
configs = sorted(stats["config"].unique(), key=lambda x: float(x.split("_")[0]))  # e.g. "1_pct_sample" → 1.0
config_labels = [cfg.replace("_", " ").replace("pct", "%").capitalize() for cfg in configs]

# --- Continuous colormap (light→dark gradient) ---
cmap = cm.get_cmap("viridis", len(configs))
colors = [cmap(i) for i in range(len(configs))]

datasets = stats["dataset"].unique()
bar_width = 0.8 / len(configs)  # keep bars compact
x = range(len(datasets))

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

# --- Plot bars for each configuration ---
for i, cfg in enumerate(configs):
    subset = stats[stats["config"] == cfg]
    ax.bar(
        [xi + (i - len(configs)/2) * bar_width for xi in x],
        subset["mean"],
        yerr=subset["std"],
        width=bar_width,
        label=config_labels[i],
        color=colors[i],
        edgecolor="black",
        linewidth=0.5,
        capsize=2,
    )

# --- Wrap labels after the first word ---
def wrap_after_first_word(label):
    label = str(label)
    parts = label.split(" ", 1)
    return "\n".join(parts) if len(parts) > 1 else label

wrapped_labels = [wrap_after_first_word(ds) for ds in datasets]
ax.set_xticks(x)
ax.set_xticklabels(wrapped_labels, fontsize=12, ha="center")

# --- Aesthetics ---
ax.set_ylabel("ARI", fontsize=12)
ax.set_ylim(0, 1.0)

# --- Legend with colormap box ---
legend = ax.legend(
    frameon=True,
    fontsize=12,
    loc="upper right",
    framealpha=0.95,
    fancybox=True,
    edgecolor="black",
    title="Dataset size",
    title_fontsize=12,
)
legend.get_frame().set_linewidth(0.6)

# --- Clean spines and grid ---
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="0.9", linestyle="-", linewidth=0.5)
ax.set_axisbelow(True)

# --- Save figure ---
save_path = os.path.join(sc.settings.figdir, "dataset_size_cell_type_ari_comparison.svg")
plt.savefig(save_path, bbox_inches="tight", format="svg")
print(f"✅ Figure saved to: {save_path}")

plt.tight_layout()
plt.show()

## 2.3 Masking Ablation

### 2.3.1 xhs1000-39b_1p (Xenium Human Skin Eczema) - Niche ID

In [ ]:
# Define dataset config
dataset_name = 'xhs1000-39b_1p'
obsm_key = 'neighborhood_emb_layer_12'
label_col = 'niche_type'

#### 2.3.1.1 0.5 Masking Ratio

In [ ]:
config_name = '50_pct_masking'
load_timestamp = '02112025_043501_781'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.1.2 0.6 Masking Ratio

In [ ]:
config_name = '60_pct_masking'
load_timestamp = '29102025_142759_837'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.1.3 0.7 Masking Ratio

In [ ]:
config_name = '70_pct_masking'
load_timestamp = '30102025_112358_100'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.1.4 0.8 Masking Ratio

In [ ]:
config_name = '80_pct_masking'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.1.5 0.9 Masking Ratio

In [ ]:
config_name = '90_pct_masking'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.3.2 chl59-8b_1p (NanoString CosMx Human Lung NSCLC) - Niche ID

In [ ]:
# Define dataset config
dataset_name = 'chl59-8b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'cell_type'

#### 2.3.2.1 0.5 Masking Ratio

In [ ]:
config_name = '50_pct_masking'
load_timestamp = '02112025_043501_781'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.2.2 0.6 Masking Ratio

In [ ]:
config_name = '60_pct_masking'
load_timestamp = '29102025_142759_837'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.2.3 0.7 Masking Ratio

In [ ]:
config_name = '70_pct_masking'
load_timestamp = '30102025_112358_100'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.2.4 0.8 Masking Ratio

In [ ]:
config_name = '80_pct_masking'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.2.5 0.9 Masking Ratio

In [ ]:
config_name = '90_pct_masking'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.3.3 xhs49-1b_1p (Xenium Human Skin Melanoma) - Niche ID

In [ ]:
# Define dataset config
dataset_name = 'xhs49-1b_1p'
obsm_key = 'neighborhood_emb_layer_12'
label_col = 'niche_name'

#### 2.3.3.1 0.5 Masking Ratio

In [ ]:
config_name = '50_pct_masking'
load_timestamp = '02112025_043501_781'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.3.2 0.6 Masking Ratio

In [ ]:
config_name = '60_pct_masking'
load_timestamp = '29102025_142759_837'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.3.3 0.7 Masking Ratio

In [ ]:
config_name = '70_pct_masking'
load_timestamp = '30102025_112358_100'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.3.4 0.8 Masking Ratio

In [ ]:
config_name = '80_pct_masking'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.3.5 0.9 Masking Ratio

In [ ]:
config_name = '90_pct_masking'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.3.4 chr78-11b_1p (ISS Cartana Human Reproductive Tract) - Niche ID

In [ ]:
# Define dataset config
dataset_name = 'chr78-11b_1p'
obsm_key = 'neighborhood_emb_layer_12'
label_col = 'broad_anatomy'

#### 2.3.4.1 0.5 Masking Ratio

In [ ]:
config_name = '50_pct_masking'
load_timestamp = '02112025_043501_781'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.4.2 0.6 Masking Ratio

In [ ]:
config_name = '60_pct_masking'
load_timestamp = '29102025_142759_837'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.4.3 0.7 Masking Ratio

In [ ]:
config_name = '70_pct_masking'
load_timestamp = '30102025_112358_100'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.4.4 0.8 Masking Ratio

In [ ]:
config_name = '80_pct_masking'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.4.5 0.9 Masking Ratio

In [ ]:
config_name = '90_pct_masking'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.3.5 xhs1000-39b_1p (Xenium Human Skin Eczema) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'xhs1000-39b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'new_annotation'

#### 2.3.5.1 0.5 Masking Ratio

In [ ]:
config_name = '50_pct_masking'
load_timestamp = '02112025_043501_781'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.5.2 0.6 Masking Ratio

In [ ]:
config_name = '60_pct_masking'
load_timestamp = '29102025_142759_837'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.5.3 0.7 Masking Ratio

In [ ]:
config_name = '70_pct_masking'
load_timestamp = '30102025_112358_100'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.5.4 0.8 Masking Ratio

In [ ]:
config_name = '80_pct_masking'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.5.5 0.9 Masking Ratio

In [ ]:
config_name = '90_pct_masking'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.3.6 chl59-8b_1p (NanoString CosMx Human Lung NSCLC) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'chl59-8b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'cell_type'

#### 2.3.6.1 0.5 Masking Ratio

In [ ]:
config_name = '50_pct_masking'
load_timestamp = '02112025_043501_781'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.6.2 0.6 Masking Ratio

In [ ]:
config_name = '60_pct_masking'
load_timestamp = '29102025_142759_837'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.6.3 0.7 Masking Ratio

In [ ]:
config_name = '70_pct_masking'
load_timestamp = '30102025_112358_100'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.6.4 0.8 Masking Ratio

In [ ]:
config_name = '80_pct_masking'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.6.5 0.9 Masking Ratio

In [ ]:
config_name = '90_pct_masking'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.3.7 xhp1016-18b_1p (Xenium Human Pancreas) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'xhp1016-18b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'Annotation_high'

#### 2.3.7.1 0.5 Masking Ratio

In [ ]:
config_name = '50_pct_masking'
load_timestamp = '02112025_043501_781'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.7.2 0.6 Masking Ratio

In [ ]:
config_name = '60_pct_masking'
load_timestamp = '29102025_142759_837'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.7.3 0.7 Masking Ratio

In [ ]:
config_name = '70_pct_masking'
load_timestamp = '30102025_112358_100'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.7.4 0.8 Masking Ratio

In [ ]:
config_name = '80_pct_masking'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.7.5 0.9 Masking Ratio

In [ ]:
config_name = '90_pct_masking'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.3.8 shp75-4b_1p (STARmap Human Placenta) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'shp75-4b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'leiden'

#### 2.3.8.1 0.5 Masking Ratio

In [ ]:
config_name = '50_pct_masking'
load_timestamp = '02112025_043501_781'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.8.2 0.6 Masking Ratio

In [ ]:
config_name = '60_pct_masking'
load_timestamp = '29102025_142759_837'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.8.3 0.7 Masking Ratio

In [ ]:
config_name = '70_pct_masking'
load_timestamp = '30102025_112358_100'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.8.4 0.8 Masking Ratio

In [ ]:
config_name = '80_pct_masking'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.8.5 0.9 Masking Ratio

In [ ]:
config_name = '90_pct_masking'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.3.9 xhk1007-41b_1p (Xenium Human Kidney RCC) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'xhk1007-41b_1p'
obsm_key = 'cell_emb_layer_12'
label_col = 'level_1_cell_type'

#### 2.3.9.1 0.5 Masking Ratio

In [ ]:
config_name = '50_pct_masking'
load_timestamp = '02112025_043501_781'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.9.2 0.6 Masking Ratio

In [ ]:
config_name = '60_pct_masking'
load_timestamp = '29102025_142759_837'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.9.3 0.7 Masking Ratio

In [ ]:
config_name = '70_pct_masking'
load_timestamp = '30102025_112358_100'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.9.4 0.8 Masking Ratio

In [ ]:
config_name = '80_pct_masking'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.9.5 0.9 Masking Ratio

In [ ]:
config_name = '90_pct_masking'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.3.10 xhb1002-75b_3p (Xenium Human Brain GBM) - Cell Type ID

In [ ]:
# Define dataset config
dataset_name = 'xhb1002-75b_3p'
obsm_key = 'cell_emb_layer_12'
label_col = 'leiden_log1pnorm_res-1'

#### 2.3.10.1 0.5 Masking Ratio

In [ ]:
config_name = '50_pct_masking'
load_timestamp = '02112025_043501_781'
read_checkpoint = 'terra-ep1.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.10.2 0.6 Masking Ratio

In [ ]:
config_name = '60_pct_masking'
load_timestamp = '29102025_142759_837'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.10.3 0.7 Masking Ratio

In [ ]:
config_name = '70_pct_masking'
load_timestamp = '30102025_112358_100'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.10.4 0.8 Masking Ratio

In [ ]:
config_name = '80_pct_masking'
load_timestamp = '01112025_081731_614'
read_checkpoint = 'terra-ep2.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

#### 2.3.10.5 0.9 Masking Ratio

In [ ]:
config_name = '90_pct_masking'
load_timestamp = '03112025_004451_930'
read_checkpoint = 'terra-ep3.pth.tar'
start_latent_leiden_resolution = 0.4

In [ ]:
compute_metrics(
    cf=config_name,
    ts=load_timestamp,
    cp=read_checkpoint,
    obs_col=label_col,
    emb_key=obsm_key,
    start_res=start_latent_leiden_resolution
)

### 2.3.11 Compare & Visualize

In [ ]:
df_xhs1000_50_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/02112025_043501_781/ep1_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_60_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/29102025_142759_837/ep2_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_70_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/30102025_112358_100/ep2_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_80_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_90_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_xhs1000-39b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_80_pct_masking['config'] = '80_pct_masking'
df_xhs1000_90_pct_masking['config'] = '90_pct_masking'

df_chl59_50_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/02112025_043501_781/ep1_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chl59_60_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/29102025_142759_837/ep2_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chl59_70_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/30102025_112358_100/ep2_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chl59_80_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chl59_90_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_chl59-8b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chl59_80_pct_masking['config'] = '80_pct_masking'
df_chl59_90_pct_masking['config'] = '90_pct_masking'

df_xhs49_50_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/02112025_043501_781/ep1_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs49_60_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/29102025_142759_837/ep2_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs49_70_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/30102025_112358_100/ep2_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs49_80_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs49_90_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_xhs49-1b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_xhs49_80_pct_masking['config'] = '80_pct_masking'
df_xhs49_90_pct_masking['config'] = '90_pct_masking'

df_chr78_50_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/02112025_043501_781/ep1_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chr78_60_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/29102025_142759_837/ep2_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chr78_70_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/30102025_112358_100/ep2_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chr78_80_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chr78_90_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_chr78-11b_1p_neighborhood_emb_leiden_rapids_avg_metrics.csv')
df_chr78_80_pct_masking['config'] = '80_pct_masking'
df_chr78_90_pct_masking['config'] = '90_pct_masking'

In [ ]:
df_xhs1000 = pd.concat([df_xhs1000_90_pct_masking, df_xhs1000_80_pct_masking, df_xhs1000_70_pct_masking, df_xhs1000_60_pct_masking, df_xhs1000_50_pct_masking], ignore_index=True)
df_xhs1000['dataset'] = 'Xenium Eczema'
df_chl59 = pd.concat([df_chl59_90_pct_masking, df_chl59_80_pct_masking, df_chl59_70_pct_masking, df_chl59_60_pct_masking, df_chl59_50_pct_masking], ignore_index=True)
df_chl59['dataset'] = 'NanoString CosMx NSCLC'
df_xhs49 = pd.concat([df_xhs49_90_pct_masking, df_xhs49_80_pct_masking, df_xhs49_70_pct_masking, df_xhs49_60_pct_masking, df_xhs49_50_pct_masking], ignore_index=True)
df_xhs49['dataset'] = 'Xenium Melanoma'
df_chr78 = pd.concat([df_chr78_90_pct_masking, df_chr78_80_pct_masking, df_chr78_70_pct_masking, df_chr78_60_pct_masking, df_chr78_50_pct_masking], ignore_index=True)
df_chr78['dataset'] = 'ISS Cartana Reproductive Tract'

In [ ]:
df_neb = pd.concat([df_xhs49, df_xhs1000, df_chl59, df_chr78], ignore_index=True)

In [ ]:
df_xhs1000_50_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/02112025_043501_781/ep1_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_60_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/29102025_142759_837/ep2_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_70_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/30102025_112358_100/ep2_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_80_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_90_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_xhs1000-39b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhs1000_80_pct_masking['config'] = '80_pct_masking'
df_xhs1000_90_pct_masking['config'] = '90_pct_masking'

df_chl59_50_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/02112025_043501_781/ep1_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_chl59_60_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/29102025_142759_837/ep2_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_chl59_70_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/30102025_112358_100/ep2_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_chl59_80_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_chl59_90_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_chl59-8b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_chl59_80_pct_masking['config'] = '80_pct_masking'
df_chl59_90_pct_masking['config'] = '90_pct_masking'

df_xhp1016_50_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/02112025_043501_781/ep1_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhp1016_60_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/29102025_142759_837/ep2_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhp1016_70_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/30102025_112358_100/ep2_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhp1016_80_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhp1016_90_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_xhp1016-18b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_xhp1016_80_pct_masking['config'] = '80_pct_masking'
df_xhp1016_90_pct_masking['config'] = '90_pct_masking'

df_shp75_50_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/02112025_043501_781/ep1_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_shp75_60_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/29102025_142759_837/ep2_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_shp75_70_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/30102025_112358_100/ep2_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_shp75_80_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_shp75_90_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_shp75-4b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
df_shp75_80_pct_masking['config'] = '80_pct_masking'
df_shp75_90_pct_masking['config'] = '90_pct_masking'

In [ ]:
df_xhs1000 = pd.concat([df_xhs1000_100_pct_sample, df_xhs1000_50_pct_sample, df_xhs1000_25_pct_sample, df_xhs1000_10_pct_sample, df_xhs1000_1_pct_sample], ignore_index=True)
df_xhs1000['dataset'] = 'Xenium Eczema'
df_chl59 = pd.concat([df_chl59_100_pct_sample, df_chl59_50_pct_sample, df_chl59_25_pct_sample, df_chl59_10_pct_sample, df_chl59_1_pct_sample], ignore_index=True)
df_chl59['dataset'] = 'NanoString CosMx NSCLC'
df_xhp1016 = pd.concat([df_xhp1016_100_pct_sample, df_xhp1016_50_pct_sample, df_xhp1016_25_pct_sample, df_xhp1016_10_pct_sample, df_xhp1016_1_pct_sample], ignore_index=True)
df_xhp1016['dataset'] = 'Xenium Developing Pancreas'
df_shp75 = pd.concat([df_shp75_100_pct_sample, df_shp75_50_pct_sample, df_shp75_25_pct_sample, df_shp75_10_pct_sample, df_shp75_1_pct_sample], ignore_index=True)
df_shp75['dataset'] = 'STARmap Healthy Placenta'

In [ ]:
df_cell = pd.concat([df_xhs1000, df_chl59, df_xhp1016, df_shp75], ignore_index=True)

In [ ]:
#df_xhk1007_50_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/02112025_043501_781/ep1_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhk1007_60_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/29102025_142759_837/ep2_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhk1007_70_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/30102025_112358_100/ep2_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhk1007_80_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhk1007_90_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_xhk1007-41b_1p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhk1007_80_pct_masking['config'] = '80_pct_masking'
#df_xhk1007_90_pct_masking['config'] = '90_pct_masking'

#df_xhb1002_50_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/02112025_043501_781/ep1_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhb1002_60_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/29102025_142759_837/ep2_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhb1002_70_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/30102025_112358_100/ep2_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhb1002_80_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/01112025_081731_614/ep2_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhb1002_90_pct_masking = pd.read_csv('/nfs/team361/sb75/terra-reproducibility/artifacts/hst_corpus_110m/03112025_004451_930/ep3_xhb1002-75b_3p_cell_emb_leiden_rapids_avg_metrics.csv')
#df_xhb1002_80_pct_masking['config'] = '80_pct_masking'
#df_xhb1002_90_pct_masking['config'] = '90_pct_masking'

In [ ]:
#df_xhk1007 = pd.concat([df_xhk1007_90_pct_masking, df_xhk1007_80_pct_masking, df_xhk1007_70_pct_masking, df_xhk1007_60_pct_masking, df_xhk1007_50_pct_masking], ignore_index=True)
#df_xhk1007['dataset'] = 'Xenium Kidney RCC'
#df_xhb1002 = pd.concat([df_xhb1002_90_pct_masking, df_xhb1002_80_pct_masking, df_xhb1002_70_pct_masking, df_xhb1002_60_pct_masking, df_xhb1002_50_pct_masking], ignore_index=True)
#df_xhb1002['dataset'] = 'Xenium Brain GBM'

In [ ]:
#df_cell = pd.concat([df_xhk1007, df_xhb1002], ignore_index=True)

In [ ]:
# --- Filter only NMI metric ---
df_nmi = df_neb[df_neb["metric"] == "NMI"]

# --- Compute mean and std per dataset × config ---
stats = (
    df_nmi.groupby(["dataset", "config"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# --- Enforce dataset order ---
desired_order = [
    "Xenium Eczema",
    "NanoString CosMx NSCLC",
    "Xenium Melanoma",
    "ISS Cartana Reproductive Tract",
]
stats["dataset"] = pd.Categorical(stats["dataset"], categories=desired_order, ordered=True)
stats = stats.sort_values("dataset").reset_index(drop=True)

# --- Configurations ---
configs = sorted(stats["config"].unique(), key=lambda x: float(x.split("_")[0]))  # e.g. "1_pct_sample" → 1.0
config_labels = [cfg.replace("_", " ").replace("pct", "%").capitalize() for cfg in configs]

# --- Continuous colormap (light→dark gradient) ---
cmap = cm.get_cmap("magma", len(configs))
colors = [cmap(i) for i in range(len(configs))]

datasets = stats["dataset"].unique()
bar_width = 0.8 / len(configs)  # keep bars compact
x = range(len(datasets))

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

# --- Plot bars for each configuration ---
for i, cfg in enumerate(configs):
    subset = stats[stats["config"] == cfg]
    ax.bar(
        [xi + (i - len(configs)/2) * bar_width for xi in x],
        subset["mean"],
        yerr=subset["std"],
        width=bar_width,
        label=config_labels[i],
        color=colors[i],
        edgecolor="black",
        linewidth=0.5,
        capsize=2,
    )

# --- Wrap labels after the first word ---
def wrap_after_first_word(label):
    label = str(label)
    parts = label.split(" ", 1)
    return "\n".join(parts) if len(parts) > 1 else label

wrapped_labels = [wrap_after_first_word(ds) for ds in datasets]
ax.set_xticks(x)
ax.set_xticklabels(wrapped_labels, fontsize=12, ha="center")

# --- Aesthetics ---
ax.set_ylabel("NMI", fontsize=12)
ax.set_ylim(0, 1.0)

# --- Legend with colormap box ---
legend = ax.legend(
    frameon=True,
    fontsize=12,
    loc="upper right",
    framealpha=0.95,
    fancybox=True,
    edgecolor="black",
    title="Dataset size",
    title_fontsize=12,
)
legend.get_frame().set_linewidth(0.6)

# --- Clean spines and grid ---
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="0.9", linestyle="-", linewidth=0.5)
ax.set_axisbelow(True)

# --- Save figure ---
save_path = os.path.join(sc.settings.figdir, "masking_ratio_niche_nmi_comparison.svg")
plt.savefig(save_path, bbox_inches="tight", format="svg")
print(f"✅ Figure saved to: {save_path}")

plt.tight_layout()
plt.show()

In [ ]:
# --- Filter only ARI metric ---
df_ari = df_neb[df_neb["metric"] == "ARI"]

# --- Compute mean and std per dataset × config ---
stats = (
    df_ari.groupby(["dataset", "config"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# --- Enforce dataset order ---
desired_order = [
    "Xenium Eczema",
    "NanoString CosMx NSCLC",
    "Xenium Melanoma",
    "ISS Cartana Reproductive Tract",
]
stats["dataset"] = pd.Categorical(stats["dataset"], categories=desired_order, ordered=True)
stats = stats.sort_values("dataset").reset_index(drop=True)

# --- Configurations ---
configs = sorted(stats["config"].unique(), key=lambda x: float(x.split("_")[0]))  # e.g. "1_pct_sample" → 1.0
config_labels = [cfg.replace("_", " ").replace("pct", "%").capitalize() for cfg in configs]

# --- Continuous colormap (light→dark gradient) ---
cmap = cm.get_cmap("magma", len(configs))
colors = [cmap(i) for i in range(len(configs))]

datasets = stats["dataset"].unique()
bar_width = 0.8 / len(configs)  # keep bars compact
x = range(len(datasets))

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

# --- Plot bars for each configuration ---
for i, cfg in enumerate(configs):
    subset = stats[stats["config"] == cfg]
    ax.bar(
        [xi + (i - len(configs)/2) * bar_width for xi in x],
        subset["mean"],
        yerr=subset["std"],
        width=bar_width,
        label=config_labels[i],
        color=colors[i],
        edgecolor="black",
        linewidth=0.5,
        capsize=2,
    )

# --- Wrap labels after the first word ---
def wrap_after_first_word(label):
    label = str(label)
    parts = label.split(" ", 1)
    return "\n".join(parts) if len(parts) > 1 else label

wrapped_labels = [wrap_after_first_word(ds) for ds in datasets]
ax.set_xticks(x)
ax.set_xticklabels(wrapped_labels, fontsize=12, ha="center")

# --- Aesthetics ---
ax.set_ylabel("ARI", fontsize=12)
ax.set_ylim(0, 1.0)

# --- Legend with colormap box ---
legend = ax.legend(
    frameon=True,
    fontsize=12,
    loc="upper right",
    framealpha=0.95,
    fancybox=True,
    edgecolor="black",
    title="Dataset size",
    title_fontsize=12,
)
legend.get_frame().set_linewidth(0.6)

# --- Clean spines and grid ---
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="0.9", linestyle="-", linewidth=0.5)
ax.set_axisbelow(True)

# --- Save figure ---
save_path = os.path.join(sc.settings.figdir, "masking_ratio_niche_ari_comparison.svg")
plt.savefig(save_path, bbox_inches="tight", format="svg")
print(f"✅ Figure saved to: {save_path}")

plt.tight_layout()
plt.show()

In [ ]:
# --- Filter only NMI metric ---
df_nmi = df_cell[df_cell["metric"] == "NMI"]

# --- Compute mean and std per dataset × config ---
stats = (
    df_nmi.groupby(["dataset", "config"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# --- Enforce dataset order ---
desired_order = [
    'Xenium Eczema',
    'NanoString CosMx NSCLC',
    'Xenium Developing Pancreas',
    'STARmap Healthy Placenta'
]
stats["dataset"] = pd.Categorical(stats["dataset"], categories=desired_order, ordered=True)
stats = stats.sort_values("dataset").reset_index(drop=True)

# --- Configurations ---
configs = sorted(stats["config"].unique(), key=lambda x: float(x.split("_")[0]))  # e.g. "1_pct_sample" → 1.0
config_labels = [cfg.replace("_", " ").replace("pct", "%").capitalize() for cfg in configs]

# --- Continuous colormap (light→dark gradient) ---
cmap = cm.get_cmap("magma", len(configs))
colors = [cmap(i) for i in range(len(configs))]

datasets = stats["dataset"].unique()
bar_width = 0.8 / len(configs)  # keep bars compact
x = range(len(datasets))

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

# --- Plot bars for each configuration ---
for i, cfg in enumerate(configs):
    subset = stats[stats["config"] == cfg]
    ax.bar(
        [xi + (i - len(configs)/2) * bar_width for xi in x],
        subset["mean"],
        yerr=subset["std"],
        width=bar_width,
        label=config_labels[i],
        color=colors[i],
        edgecolor="black",
        linewidth=0.5,
        capsize=2,
    )

# --- Wrap labels after the first word ---
def wrap_after_first_word(label):
    label = str(label)
    parts = label.split(" ", 1)
    return "\n".join(parts) if len(parts) > 1 else label

wrapped_labels = [wrap_after_first_word(ds) for ds in datasets]
ax.set_xticks(x)
ax.set_xticklabels(wrapped_labels, fontsize=12, ha="center")

# --- Aesthetics ---
ax.set_ylabel("NMI", fontsize=12)
ax.set_ylim(0, 1.0)

# --- Legend with colormap box ---
legend = ax.legend(
    frameon=True,
    fontsize=12,
    loc="upper right",
    framealpha=0.95,
    fancybox=True,
    edgecolor="black",
    title="Masking ratio",
    title_fontsize=12,
)
legend.get_frame().set_linewidth(0.6)

# --- Clean spines and grid ---
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="0.9", linestyle="-", linewidth=0.5)
ax.set_axisbelow(True)

# --- Save figure ---
save_path = os.path.join(sc.settings.figdir, "masking_ratio_cell_type_nmi_comparison.svg")
plt.savefig(save_path, bbox_inches="tight", format="svg")
print(f"✅ Figure saved to: {save_path}")

plt.tight_layout()
plt.show()

In [ ]:
# --- Filter only ARI metric ---
df_ari = df_cell[df_cell["metric"] == "ARI"]

# --- Compute mean and std per dataset × config ---
stats = (
    df_ari.groupby(["dataset", "config"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

# --- Enforce dataset order ---
desired_order = [
    'Xenium Eczema',
    'NanoString CosMx NSCLC',
    'Xenium Developing Pancreas',
    'STARmap Healthy Placenta'
]
stats["dataset"] = pd.Categorical(stats["dataset"], categories=desired_order, ordered=True)
stats = stats.sort_values("dataset").reset_index(drop=True)

# --- Configurations ---
configs = sorted(stats["config"].unique(), key=lambda x: float(x.split("_")[0]))  # e.g. "1_pct_sample" → 1.0
config_labels = [cfg.replace("_", " ").replace("pct", "%").capitalize() for cfg in configs]

# --- Continuous colormap (light→dark gradient) ---
cmap = cm.get_cmap("magma", len(configs))
colors = [cmap(i) for i in range(len(configs))]

datasets = stats["dataset"].unique()
bar_width = 0.8 / len(configs)  # keep bars compact
x = range(len(datasets))

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=300)

# --- Plot bars for each configuration ---
for i, cfg in enumerate(configs):
    subset = stats[stats["config"] == cfg]
    ax.bar(
        [xi + (i - len(configs)/2) * bar_width for xi in x],
        subset["mean"],
        yerr=subset["std"],
        width=bar_width,
        label=config_labels[i],
        color=colors[i],
        edgecolor="black",
        linewidth=0.5,
        capsize=2,
    )

# --- Wrap labels after the first word ---
def wrap_after_first_word(label):
    label = str(label)
    parts = label.split(" ", 1)
    return "\n".join(parts) if len(parts) > 1 else label

wrapped_labels = [wrap_after_first_word(ds) for ds in datasets]
ax.set_xticks(x)
ax.set_xticklabels(wrapped_labels, fontsize=12, ha="center")

# --- Aesthetics ---
ax.set_ylabel("ARI", fontsize=12)
ax.set_ylim(0, 1.0)

# --- Legend with colormap box ---
legend = ax.legend(
    frameon=True,
    fontsize=12,
    loc="upper right",
    framealpha=0.95,
    fancybox=True,
    edgecolor="black",
    title="Dataset size",
    title_fontsize=12,
)
legend.get_frame().set_linewidth(0.6)

# --- Clean spines and grid ---
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="0.9", linestyle="-", linewidth=0.5)
ax.set_axisbelow(True)

# --- Save figure ---
save_path = os.path.join(sc.settings.figdir, "masking_ratio_cell_type_ari_comparison.svg")
plt.savefig(save_path, bbox_inches="tight", format="svg")
print(f"✅ Figure saved to: {save_path}")

plt.tight_layout()
plt.show()